In [1]:
# Step 1
# Packages (numpy, CoolProp, matplotlib) and the output helpers say() / rep()
# are in the file SETUP_FILE, next to this notebook. This cell loads them.
SETUP_FILE = 'lunar_setup.py'
with open(SETUP_FILE) as _f:
    exec(compile(_f.read(), SETUP_FILE, 'exec'))

In [2]:
# Step 2
# Inputs: every input value is in the file INPUTS_FILE, next to this notebook.
# To run another case, edit that file (or point INPUTS_FILE to a copy of it) and run
# the notebook from Step 1. This cell reads it, checks it and logs a summary.
# Start every run with an empty log and report, so re-running the notebook from the
# inputs never repeats lines in the PDF (Step 12 also clears REPORT before it writes).
RUN_LOG.clear()
REPORT.clear()
INPUTS_FILE = 'lunar_inputs.py'
with open(INPUTS_FILE) as _f:
    exec(compile(_f.read(), INPUTS_FILE, 'exec'))
say(f"Inputs read from {INPUTS_FILE}")

In [3]:
# Step 3
# Regolith properties (lunar_regolith.py) and the TES block model with two tube sets
# (lunar_tes.py), both next to this notebook.
for _file in ('lunar_regolith.py', 'lunar_tes.py'):
    with open(_file) as _f:
        exec(compile(_f.read(), _file, 'exec'))
say(f"Regolith properties ({regolith_type}) and TES block model loaded")

In [4]:
# Step 4
# Simple sCO2 Brayton cycle (Appendix Table A, Eq. 1-10) and its (T1, TIT) table are in
# the file SIMPLE_FILE, next to this notebook. This cell loads them.
SIMPLE_FILE = 'lunar_cycle_simple.py'
with open(SIMPLE_FILE) as _f:
    exec(compile(_f.read(), SIMPLE_FILE, 'exec'))

In [5]:
# Step 5
# Recuperated sCO2 Brayton cycle (Appendix Table A, Eq. 11-20) and its (T1, TIT) table are
# in the file RECUP_FILE, next to this notebook. It reuses the compressor, turbine and
# grids from Step 4, so run Step 4 first.
RECUP_FILE = 'lunar_cycle_recuperated.py'
with open(RECUP_FILE) as _f:
    exec(compile(_f.read(), RECUP_FILE, 'exec'))

In [6]:
# Step 6
# Plant model (layout of Fig. 2: TES gas loop, power cycle, coolant loop, habitat, radiator)
# is in the file PLANT_FILE, next to this notebook. This cell loads it: it defines
# plant_point() and the loop/radiator functions, adds the loop flow ratios to the cycle
# maps, finds the usable TES range and sizes the radiator for the TES floor and the PHX.
PLANT_FILE = 'lunar_plant.py'
with open(PLANT_FILE) as _f:
    exec(compile(_f.read(), PLANT_FILE, 'exec'))
say(f"Plant model read from {PLANT_FILE}")

In [7]:
# Step 7
# Discharge table (night): cycle and plant efficiency, heat from the TES and TES - TIT gap
# every 25 K of TES temperature, for both cycles. The code is in the file DISCHARGE_FILE,
# next to this notebook; the table goes to the run log in the PDF.
DISCHARGE_FILE = 'lunar_discharge_table.py'
with open(DISCHARGE_FILE) as _f:
    exec(compile(_f.read(), DISCHARGE_FILE, 'exec'))

In [8]:
# Step 8
# Solar input (sun angle, heliostat cosine factor, receiver losses) and the radiator
# environment (lunar surface and sink temperatures) are in the file SOLAR_FILE, next to
# this notebook. This cell loads them; the quick-look table goes to the run log.
SOLAR_FILE = 'lunar_solar.py'
with open(SOLAR_FILE) as _f:
    exec(compile(_f.read(), SOLAR_FILE, 'exec'))

In [9]:
# Step 9
# Hour-by-hour simulation of the TES energy balance (day and night): simulate() is in the
# file SIMULATE_FILE, next to this notebook. This cell loads it; Step 10 uses it to size
# the plant and to run the final lunar cycle.
SIMULATE_FILE = 'lunar_simulate.py'
with open(SIMULATE_FILE) as _f:
    exec(compile(_f.read(), SIMULATE_FILE, 'exec'))

In [10]:
# Step 10a
SIZING_FILE = 'lunar_sizing.py'
with open(SIZING_FILE) as _f:
    exec(compile(_f.read(), SIZING_FILE, 'exec'))
say("Sizing functions loaded")

In [11]:
# Step 10b
# Size every cycle (lunar_sizing.py, Step 10a): TES mass, collector area and the final
# month -> R[ct]. Uses the radiator Step 6 sized for the store floor.
def _size(ct):
    return dict(ct=ct, R=size_system(ct))

_radiator_from_step6()
_t0 = _time.time()
_res, _how = _run_part(_size)
R = {_r['ct']: _r['R'] for _r in _res}
_part_time('10b (sizing)', _t0, _how)

Step 10b (sizing): 2.5 min (one cycle after the other)


In [12]:
# Step 10c
FINISH_FILE = 'lunar_finish.py'
with open(FINISH_FILE) as _f:
    exec(compile(_f.read(), FINISH_FILE, 'exec'))

Step 10c (finish): 0.1 min (one cycle after the other)


In [13]:
# Step 11
# --- Energy balances over the final month, for each cycle (used by the report) ---
#   TES  : heat in from the charging gas - heat out to the discharging gas - insulation loss
#          - change in stored energy = 0
#   sun  : heat into the receiver gas = heat stored + heat curtailed (defocused)
#   PHX  : heat out of the bricks vs heat into the cycle (differs only by the small change of
#          the end-A gas temperature within a step)
#   cycle: heat in - shaft power - heat out = 0, every step the cycle runs
for ct in cycles:
    cy = R[ct]['cycle']
    _s = slice(0, -1)                                  # steps, not end point
    E_solar   = np.sum(cy['Q_solar'][_s]) * dt         # kWh, net heat into the receiver gas
    E_curtail = np.sum(cy['Q_curtail'][_s]) * dt
    E_ch      = np.sum(cy['Q_ch'][_s]) * dt            # into the bricks (charging loop)
    E_dis     = np.sum(cy['Q_dis'][_s]) * dt           # out of the bricks (discharging loop)
    E_loss    = np.sum(cy['Q_loss'][_s]) * dt          # through the insulation
    E_drawn   = np.sum(cy['Q_in'][_s]) * dt            # into the cycle at the PHX
    E_change  = cy['E_TES'][-1] - cy['E_TES'][0]       # stored energy, end - start
    R[ct]['balance'] = (E_ch - E_dis - E_loss) - E_change                       # kWh, ~0
    R[ct]['sun_balance'] = E_solar - E_ch - E_curtail                           # kWh, ~0
    R[ct]['phx_diff'] = 100 * (E_dis - E_drawn) / max(E_drawn, 1e-9)            # %, small
    _on = cy['running']                                # only steps the cycle runs
    R[ct]['cycle_balance'] = (np.max(np.abs(cy['Q_in'][_on] - cy['W_net'][_on] - cy['Q_out'][_on]))
                              if _on.any() else 0.0)  # kW, ~0
    say(f"{ct:>11}: TES balance {R[ct]['balance']:.2e} kWh (of {E_ch:.0f} kWh charged), "
        f"sun balance {R[ct]['sun_balance']:.2e} kWh, bricks vs PHX {R[ct]['phx_diff']:+.2f} %, "
        f"cycle balance {R[ct]['cycle_balance']:.2e} kW")

In [14]:
# Step 12
# Report, figures, references and the results PDF -- all in one go. The code is in the file
# REPORT_FILE, next to this notebook (Part A text report, Part B AIAA figures saved to
# figures_AIAA/, Part C references, Part D the single results PDF OUT_PDF).
# Each figure is rendered once and used for both its PNG file and its PDF page.
REPORT_FILE = 'lunar_report.py'
with open(REPORT_FILE) as _f:
    exec(compile(_f.read(), REPORT_FILE, 'exec'))

Saved Lunar_TES_sCO2_results.pdf: 32 pages (summary, report, 27 figures, references)
